In [0]:
with warehouse_usage as (select 
  usage_date,
  usage_metadata.warehouse_id as warehouse_id,
  sku_name, 
  sum(usage_quantity) as total_usage, 
  identity_metadata.owned_by as user
from system.billing.usage
where usage_date >= current_date() - interval 7 days
and usage_metadata.warehouse_id is not null
group by usage_date, usage_metadata.warehouse_id,identity_metadata.owned_by ,sku_name, user
order by total_usage desc
) 
SELECT
  n.usage_date, 
  n.warehouse_id,
  n.sku_name,
  n.user,
  (n.total_usage / t.total_usage) * t.total_cost AS warehouse_run_cost
FROM
  warehouse_usage n CROSS JOIN prd.value_stream_monitoring.sku_usage_cost t
WHERE
  n.sku_name = t.sku
ORDER BY
  warehouse_run_cost DESC
